In [1]:
from xgboost import XGBRanker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [2]:
ltr_df = pd.read_parquet("../data/processed/ltr_training_dataset.parquet")
ltr_df

,name,categories,city,state,stars,review_count,combined_reviews,checkin_count,search_text,tfidf_score,query,bm25_score,query_in_name,query_in_category,label
0,Devour Indy,"Festivals, Arts & Entertainment",Indianapolis,IN,3.5,5,This is a great way to eat at that restaurant ...,5.0,"Devour Indy Festivals, Arts & Entertainment Th...",0.215237,restaurants,1.043466,0,0,1
1,New Chong Weh,"Restaurants, Chinese",Saint Louis,MO,3.0,5,I ordered a half order of shrimp fried rice wi...,1.0,"New Chong Weh Restaurants, Chinese I ordered a...",0.178316,restaurants,0.965055,0,1,1
2,Restaurant Week,Local Flavor,Philadelphia,PA,3.5,5,"Big ripoff! Lousy portions, limited menus. Big...",0.0,Restaurant Week Local Flavor Big ripoff! Lousy...,0.143985,restaurants,1.014420,0,0,1
3,The Gulch,"Restaurants, Shopping, Arts & Entertainment, L...",Nashville,TN,4.0,15,This is where I have spent most of my time whe...,49.0,"The Gulch Restaurants, Shopping, Arts & Entert...",0.141232,restaurants,1.047412,0,1,1
4,Opa Gyros Berlin,"Greek, Restaurants",Berlin,NJ,5.0,5,I have eaten at many Greek restaurants across ...,1.0,"Opa Gyros Berlin Greek, Restaurants I have eat...",0.137451,restaurants,0.991020,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,Blinds To Go,"Shades & Blinds, Home Services, Interior Design",Abington,PA,2.5,15,The store does not believe in customer satisfa...,7.0,"Blinds To Go Shades & Blinds, Home Services, I...",0.007479,eyelash service,6.099332,0,0,0
2996,Playa Bowls,"Acai Bowls, Food, Juice Bars & Smoothies, Poke",West Chester,PA,4.5,17,"Went to Playa Bowls West Chester tonight, loca...",11.0,"Playa Bowls Acai Bowls, Food, Juice Bars & Smo...",0.001248,eyelash service,3.967480,0,0,0
2997,Home Goods,"Shopping, Fashion, Department Stores",Langhorne,PA,3.5,7,Very hit or miss. I have come several times an...,54.0,"Home Goods Shopping, Fashion, Department Store...",0.002713,eyelash service,4.290105,0,0,0
2998,Buff Exteriors,"Gutter Services, Roofing, Home Services, Contr...",Saint Louis,MO,3.0,5,We called Buff after seeing them recommended o...,0.0,"Buff Exteriors Gutter Services, Roofing, Home ...",0.002064,eyelash service,4.317642,0,0,0


In [3]:
feature_cols = [
    "tfidf_score",
    "bm25_score",
    "stars",
    "review_count",
    "checkin_count",
    "query_in_name",
    "query_in_category"
]
target_col = "label"

In [4]:
ltr_df = ltr_df.dropna(subset=feature_cols + ["query"])

In [5]:
scaler = StandardScaler()
ltr_df[feature_cols] = scaler.fit_transform(ltr_df[feature_cols])

In [12]:
import joblib
joblib.dump(scaler, "../models/feature_scaler.pkl")


['../models/feature_scaler.pkl']

In [6]:
queries = ltr_df["query"].unique()
train_q, test_q = train_test_split(queries, test_size=0.2, random_state=42)

In [7]:
train_df = ltr_df[ltr_df["query"].isin(train_q)]
test_df = ltr_df[ltr_df["query"].isin(test_q)]

In [8]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]
group_train = train_df.groupby("query").size().tolist()

X_test = test_df[feature_cols]
y_test = test_df[target_col]
group_test = test_df.groupby("query").size().tolist()

In [9]:
model = XGBRanker(
    objective="rank:pairwise",
    learning_rate=0.1,
    n_estimators=100,
    max_depth=6,
    verbosity=1,
    random_state=42
)

model.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_test, y_test)],
    eval_group=[group_test],
    eval_metric="ndcg",
    early_stopping_rounds=10,
    verbose=True
)

[0]	validation_0-ndcg:0.99990
[1]	validation_0-ndcg:0.99986
[2]	validation_0-ndcg:0.99986
[3]	validation_0-ndcg:0.99986
[4]	validation_0-ndcg:0.99986
[5]	validation_0-ndcg:0.99986
[6]	validation_0-ndcg:0.99986
[7]	validation_0-ndcg:0.99986
[8]	validation_0-ndcg:0.99986
[9]	validation_0-ndcg:0.99986
[10]	validation_0-ndcg:0.99986


c:\Users\Owner\.conda\envs\all4gpu\lib\site-packages\xgboost\sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
c:\Users\Owner\.conda\envs\all4gpu\lib\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None, learning_rate=0.1,
          max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
          max_delta_step=None, max_depth=6, max_leaves=None,
          min_child_weight=None, missing=nan, monotone_constraints=None,
          multi_strategy=None, n_estimators=100, n_jobs=None,
          num_parallel_tree=None, objective='rank:pairwise', ...)

In [11]:
model.save_model("../models/ltr_model.json")